# Use Case A: Automatic Data Integration Across Multiple Knowledge Bases

Managing and comparing optimisation experiments from different research groups typically requires significant manual effort: collecting data from different sources, converting between formats, and resolving naming inconsistencies. The COMON knowledge base addresses this by design.

Because all experiments are annotated using the same shared vocabulary, data from different sources can be retrieved and integrated automatically. No manual preprocessing or format conversion is needed. This also means that a knowledge base maintained by a different research group, as long as it uses the same COMON vocabulary, would integrate just as seamlessly.

To demonstrate this, the COMON KB distributes its data across **three separate SPARQL endpoints**, each representing a dataset that could be hosted by a different group:

| Endpoint | Contents |
|---|---|
| **COMON** | All experiment metadata + execution data for 41 experiments (14 algorithms) |
| **COMON-GDE3** | Execution data for GDE3 on 3 benchmark problems (contributed separately) |
| **COMON-DMulti-MADS** | Execution data for DMulti-MADS on 3 benchmark problems (contributed separately) |

The goal of this use case is to retrieve the final performance of all **47 experiments** across three benchmark problems, using the same SPARQL query on each endpoint and merging the results automatically into a single comparison table.

> **Note.** In practice, analysts would not need to write SPARQL queries directly. A specialised platform for constrained multi-objective optimisation could run these queries in the background and expose a simple interface. Here we show the full process step by step to make clear how the ontology enables automatic integration.

## Setup

In [1]:
import requests
import pandas as pd
import numpy as np
import re
from io import StringIO

# ── SPARQL endpoints ──────────────────────────────────────────────────────────
ENDPOINT_COMON        = "http://semanticannotations.ijs.si:3030/COMON/sparql"
ENDPOINT_GDE3         = "http://semanticannotations.ijs.si:3030/COMON_gde3/sparql"
ENDPOINT_DMULTI_MADS  = "http://semanticannotations.ijs.si:3030/COMON_dmulti-mads/sparql"

ALL_ENDPOINTS = {
    "COMON":             ENDPOINT_COMON,
    "COMON-GDE3":        ENDPOINT_GDE3,
    "COMON-DMulti-MADS": ENDPOINT_DMULTI_MADS,
}

def sparql_query(query: str, endpoint: str) -> pd.DataFrame:
    """Send a SPARQL SELECT query, return results as a DataFrame."""
    response = requests.get(
        endpoint,
        params={"query": query},
        headers={"Accept": "text/csv"},
        timeout=180,
    )
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))

print("Setup complete.")

Setup complete.


## Step 1: Explore the Experiment Landscape

All experiment metadata is stored in the main COMON endpoint: which algorithm implementation was used, which benchmark problem it was applied to, and at what granularity the data was recorded. We query this endpoint first to get a full picture of the 47 experiments available.

In [2]:
query_metadata = """
PREFIX comon:   <http://w3id.org/COMON/>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?expExec ?expImplLabel ?problemLabel ?logLevel
WHERE {
  ?expExec rdf:type           comon:COMON_000014 ;
           obo:OBI_0000308    ?expImpl ;
           comon:COMON_000081 ?logLevelInd .
  ?logLevelInd rdfs:label     ?logLevel .

  ?expImpl rdfs:label         ?expImplLabel .

  ?exp obo:OBI_0000297        ?expImpl ;
       obo:OBI_0000293        ?problem .
  ?problem rdf:type           comon:COMON_000018 ;
           rdfs:label         ?problemLabel .
}
ORDER BY ?expImplLabel ?problemLabel
"""

df_meta = sparql_query(query_metadata, ENDPOINT_COMON)
print(f"Found {len(df_meta)} experiments in the COMON knowledge base.")

Found 47 experiments in the COMON knowledge base.


In [3]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 60)
df_meta[["expImplLabel", "problemLabel", "logLevel"]]

,expImplLabel,problemLabel,logLevel
0,bico_cobi1_final_platemo_experiment_implementation,cobi1_problem,final
1,bico_cre31_final_platemo_experiment_implementation,cre31_problem,final
2,bico_mw1_final_platemo_experiment_implementation,mw1_problem,final
3,ccmo_cobi1_gen_platemo_experiment_implementation,cobi1_problem,generation
4,ccmo_cre31_gen_platemo_experiment_implementation,cre31_problem,generation
5,ccmo_mw1_gen_platemo_experiment_implementation,mw1_problem,generation
6,como-cma_cobi1_eval_pycma_experiment_implementation,cobi1_problem,evaluation
7,como-cma_mw1_eval_pycma_experiment_implementation,mw1_problem,evaluation
8,csop-rs_cobi1_eval_author_experiment_implementation,cobi1_problem,evaluation
9,csop-rs_cre31_eval_author_experiment_implementation,cre31_problem,evaluation


### Observations

The 47 experiments cover three benchmark problems (COBI1, CRE31, MW1) and 16 algorithm implementations from diverse platforms: PlatEMO, pymoo, jMetal, jMetalPy, PyCMA, PyMOODE, SciPy, and custom author implementations. Experiments use different log levels: some record data only at the end of the run (*final*), some once per generation (*generation*), and some once per solution evaluation (*evaluation*).

Despite these differences in platform and logging granularity, all experiments share the same COMON vocabulary for metadata and performance indicator annotations. This shared vocabulary is what makes automatic integration possible.

## Step 2: Locate the Execution Data

While the metadata lives entirely in the main COMON endpoint, the actual execution data — the per-run performance indicator values — is distributed across all three endpoints. We query each endpoint for the number of experiment executions it contains to confirm this distribution.

In [4]:
query_count = """
PREFIX comon: <http://w3id.org/COMON/>
PREFIX obo:   <http://purl.obolibrary.org/obo/>
PREFIX rdf:   <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT (COUNT(DISTINCT ?expExec) AS ?numExperiments)
WHERE {
  ?expExec obo:BFO_0000051 ?algExec .
  ?algExec rdf:type comon:COMON_000006 .
}
"""

print(f"{'Endpoint':<25}  {'Experiments with data':>22}")
print("-" * 50)
total = 0
for name, endpoint in ALL_ENDPOINTS.items():
    df_count = sparql_query(query_count, endpoint)
    n = int(df_count["numExperiments"].iloc[0])
    total += n
    print(f"{name:<25}  {n:>22}")
print("-" * 50)
print(f"{'Total':<25}  {total:>22}")

Endpoint                    Experiments with data
--------------------------------------------------
COMON                                          41
COMON-GDE3                                      3
COMON-DMulti-MADS                               3
--------------------------------------------------
Total                                          47


The COMON endpoint holds execution data for 41 experiments. GDE3 and DMulti-MADS data were contributed separately and live in their own endpoints, each covering 3 experiments.

Because all three use the same COMON vocabulary, the same SPARQL query works on each endpoint and the results can be merged without any manual adjustment. This is exactly the scenario where a shared ontology adds value: a dataset from a different group, annotated with the same vocabulary, integrates automatically regardless of where it is hosted.

## Step 3: Query Final Performance Data from All Three Endpoints

We now retrieve the final performance value for each experiment: one value per run for each of the four performance indicators (HV, IGD+, ICMOP, CV). The *final* value is the last recorded measurement in the run. For experiments with final-only logging this is the single stored value. For experiments with generation- or evaluation-level logging, it is the value at the last recorded point.

The same SPARQL query is sent to all three endpoints. A subquery identifies the last recorded evaluation per run by taking the maximum evaluation index. The outer query then retrieves the four indicator values at that point. Results from all three endpoints are concatenated automatically.

In [5]:
query_final_data = """
PREFIX comon:   <http://w3id.org/COMON/>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dc:      <http://purl.org/dc/elements/1.1/>
PREFIX ontoopt: <http://w3id.org/ontoopt/>

SELECT ?expExec ?run ?hv ?igdPlus ?icmop ?cv
WHERE {
  # \u2500\u2500 Subquery: last eval per (experiment execution, run) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
  {
    SELECT ?expExec ?run (MAX(?evalId) AS ?lastEval)
    WHERE {
      ?expExec obo:BFO_0000051  ?algExec .
      ?algExec rdf:type         comon:COMON_000006 ;
               dc:identifier    ?run .
      ?algExec obo:BFO_0000051  ?iterExec .
      ?iterExec rdf:type        comon:COMON_000011 ;
                obo:BFO_0000051 ?solEval .
      ?solEval rdf:type         comon:COMON_000012 ;
               dc:identifier    ?evalId .
    }
    GROUP BY ?expExec ?run
  }

  # \u2500\u2500 Outer: navigate to the specific evaluation at lastEval \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
  ?expExec obo:BFO_0000051  ?algExec .
  ?algExec rdf:type         comon:COMON_000006 ;
           dc:identifier    ?run .
  ?algExec obo:BFO_0000051  ?iterExec .
  ?iterExec rdf:type        comon:COMON_000011 ;
            obo:BFO_0000051 ?solEval .
  ?solEval rdf:type         comon:COMON_000012 ;
           dc:identifier    ?lastEval .

  OPTIONAL {
    ?solEval obo:BFO_0000051 ?piExecHV .
    ?piExecHV rdf:type        comon:COMON_000007 ;
              rdfs:label      ?hvLabel ;
              obo:OBI_0000299 ?hvDatum .
    FILTER(CONTAINS(?hvLabel, "_HV-"))
    ?hvDatum ontoopt:has_value ?hv .
  }
  OPTIONAL {
    ?solEval obo:BFO_0000051 ?piExecIGD .
    ?piExecIGD rdf:type       comon:COMON_000007 ;
               rdfs:label     ?igdLabel ;
               obo:OBI_0000299 ?igdDatum .
    FILTER(CONTAINS(?igdLabel, "_IGDPlus-"))
    ?igdDatum ontoopt:has_value ?igdPlus .
  }
  OPTIONAL {
    ?solEval obo:BFO_0000051 ?piExecICMOP .
    ?piExecICMOP rdf:type      comon:COMON_000007 ;
                 rdfs:label    ?icmopLabel ;
                 obo:OBI_0000299 ?icmopDatum .
    FILTER(CONTAINS(?icmopLabel, "_ICMOP-"))
    ?icmopDatum ontoopt:has_value ?icmop .
  }
  OPTIONAL {
    ?solEval obo:BFO_0000051 ?piExecCV .
    ?piExecCV rdf:type        comon:COMON_000007 ;
              rdfs:label      ?cvLabel ;
              obo:OBI_0000299 ?cvDatum .
    FILTER(CONTAINS(?cvLabel, "_CV-"))
    ?cvDatum ontoopt:has_value ?cv .
  }
}
ORDER BY ?expExec ?run
"""

dfs = []
for name, endpoint in ALL_ENDPOINTS.items():
    print(f"Querying {name}...")
    df = sparql_query(query_final_data, endpoint)
    df["source"] = name
    print(f"  Returned {len(df):,} rows.")
    dfs.append(df)

df_data = pd.concat(dfs, ignore_index=True)
for col in ["hv", "igdPlus", "icmop", "cv"]:
    df_data[col] = pd.to_numeric(df_data[col], errors="coerce")
df_data["run"] = pd.to_numeric(df_data["run"], errors="coerce").astype(int)

print(f"\nTotal rows across all endpoints: {len(df_data):,}")
print(f"Unique experiment executions   : {df_data['expExec'].nunique()}")


Querying COMON...
  Returned 767 rows.
Querying COMON-GDE3...
  Returned 45 rows.
Querying COMON-DMulti-MADS...
  Returned 45 rows.

Total rows across all endpoints: 857
Unique experiment executions   : 47


## Step 4: Build the Summary Table

After obtaining all the experiment data, we then compute the **median** across the 15 independent runs for each performance indicator. The result is organised as a pivot table where rows are algorithm implementations and columns group the four indicators by benchmark problem. 

In [13]:
import re

def parse_expExec(iri):
    local = iri.split("/")[-1].replace("_experiment_execution", "")
    problem = next((p for p in ["cobi1", "cre31", "mw1"] if f"_{p}_" in local), None)
    algo = re.sub(r"_(cobi1|cre31|mw1)", "", local)
    algo = re.sub(r"_(final|gen|eval)", "", algo)
    return algo, problem

df_data[["Algorithm", "Problem"]] = pd.DataFrame(
    df_data["expExec"].apply(parse_expExec).tolist(),
    index=df_data.index,
)

# ── Problem order: sorted from what is actually in the data ───────────────────
problems = sorted(df_data["Problem"].dropna().unique())

# ── Indicator order: follows the SPARQL SELECT clause (?hv ?igdPlus ?icmop ?cv)
PI_COLS  = ["hv", "igdPlus", "icmop", "cv"]
PI_NAMES = ["HV", "IGD+", "ICMOP", "CV"]

# ── Build pivot problem-by-problem, then concatenate ─────────────────────────
parts = []
for prob in problems:
    sub = (
        df_data[df_data["Problem"] == prob]
        .groupby("Algorithm")[PI_COLS]
        .median()
    )
    sub.columns = pd.MultiIndex.from_product([[prob], PI_NAMES])
    parts.append(sub)

pivot = pd.concat(parts, axis=1).sort_index()
# pivot.index.name = None


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
pivot.round(4)

cobi1                       cre31                        \
                         HV   IGD+  ICMOP     CV     HV   IGD+   ICMOP     CV   
Algorithm                                                                       
bico_platemo         0.2736 0.2041 0.2736 0.0000 0.4855 0.0487  0.4855 0.0000   
ccmo_platemo         0.7973 0.0226 0.7973 0.0000 0.4742 0.0531  0.4742 0.0000   
como-cma_pycma       0.8056 0.0510 0.8056 0.0000    NaN    NaN     NaN    NaN   
csop-rs_author       0.8530 0.0046 0.8530 0.0000 0.4398 0.0946  0.4398 0.0000   
dmulti-mads_bigeon21 0.8056 0.0745 0.8056 0.0000 0.5714 0.0061  0.5714 0.0000   
gde3_pymoode         0.8549 0.0025 0.8549 0.0000 0.5713 0.0057  0.5713 0.0000   
ils_author           0.1628 0.2992 0.1628 0.0000 0.0140 0.7166  0.0140 0.0000   
mccmo_platemo        0.7651 0.0184 0.7651 0.0000 0.4796 0.0512  0.4796 0.0000   
mo-cma_pycma         0.6284 0.0697 0.6284 0.0000 0.0000 0.7354 -1.4938 0.4938   
mo-cobyla_scipy      0.1476 0.2576 0.1476 0.0000 0.2482 0.2211  0.2482 0.0000   
moeadd_platemo       0.1035 0.4632 0.1035 0.0000 0.1597 0.3347  0.1597 0.0000   
nsga-ii_jmetal       0.0420 0.2847 0.0420 0.0000 0.4328 0.0724  0.4328 0.0000   
nsga-ii_pymoo        0.8516 0.0034 0.8516 0.0000 0.5668 0.0074  0.5668 0.0000   
rs_author            0.6457 0.1200 0.6457 0.0000 0.2107 0.2732  0.2107 0.0000   
sms-emoa_pymoo       0.8382 0.0048 0.8382 0.0000 0.5056 0.0368  0.5056 0.0000   
spea2_jmetalpy       0.0415 0.2848 0.0415 0.0000 0.4642 0.0582  0.4642 0.0000   

                        mw1                        
                         HV   IGD+   ICMOP     CV  
Algorithm                                          
bico_platemo         0.4500 0.0111  0.4500 0.0000  
ccmo_platemo         0.0000 0.0111 -1.0175 0.0175  
como-cma_pycma       0.0000    NaN -1.4328 0.4328  
csop-rs_author       0.0000    NaN -1.7825 0.7825  
dmulti-mads_bigeon21 0.0000 0.0068 -1.1384 0.1384  
gde3_pymoode         0.4694 0.0000  0.4694 0.0000  
ils_author           0.0000    NaN -1.7181 0.7181  
mccmo_platemo        0.0000 0.0073 -1.0523 0.0523  
mo-cma_pycma         0.0000    NaN -1.2912 0.2912  
mo-cobyla_scipy      0.0000    NaN -1.6750 0.6750  
moeadd_platemo       0.0000    NaN -1.0523 0.0523  
nsga-ii_jmetal       0.0000 0.0037 -1.0175 0.0175  
nsga-ii_pymoo        0.0000 0.0400 -1.0646 0.0646  
rs_author            0.0000    NaN -1.7763 0.7763  
sms-emoa_pymoo       0.4682 0.0014  0.4682 0.0000  
spea2_jmetalpy       0.0000    NaN -1.0505 0.0505

## Summary

This use case showed how COMON enables automatic data integration across multiple knowledge bases. A single metadata query to the main COMON endpoint gave a complete picture of all 47 experiments. The same execution data query was then sent to all three endpoints, and the results were concatenated without any manual adjustment.

The final table covers 16 algorithm implementations across three benchmark problems (COBI1, CRE31, MW1), with median final values for four performance indicators (HV, IGD+, ICMOP, CV) across 15 independent runs per experiment. The missing entry for CoMo-CMA on CRE31 reflects that this experiment was not conducted, not a gap in the integration.

The shared COMON vocabulary is the key enabler: because every algorithm, experiment, execution, and performance indicator is described using the same ontology terms, data from different platforms, tools, and research groups can be queried and merged automatically. Any additional dataset annotated with the same vocabulary would integrate into this table with no extra effort.